# Manual multiple stain alignment

This notebook is for manually align blocks that failed to align with superglue. It assumes that affine or rigid has been attempted

---

## Follow this procedure

### A. Initialize the aligner with a block and visualize previous alignment result.
```
aligner = ManualBlockAligner(cf)
aligner.init_one_block(block="SU-17-16587.A1")
aligner.commit_save_block(do_save=False)
```
### B. Use one of the following options to align:
  - Visualize a guess
```
aligner.set_align_pair(he_i=0, ihc_i=18, do_viz=True)
aligner.visualize_transform({
    'rotation':    -85,
    'scale_x':      1,
    'scale_y':      1,
    'shear':        0,
    'tx':         60,
    'ty':       1250
})
aligner.commit_slide_pair()
```
  - Commit directly using using the inverse of the current transform it it looks good
```
aligner.set_align_pair(he_i=1, ihc_i=0, do_viz=False)
guess = decompose_affine_matrix(inverse_affine_transform(aligner.align_results["matrices"][0,1][:2, :]))
aligner.commit_slide_pair(guess=guess, do_viz=True)
```
  - Chain transform using some other third slide
```
aligner.set_align_pair(he_i=0, ihc_i=6, do_viz=False)
aligner.chain_align_with_intermed(intermed_i=1)
aligner.commit_slide_pair(do_viz=True)
```
### C. Once all slide pairs look good, commit and save the entire block
```
aligner.commit_save_block()
```

## Practical note
Now we have a gui to do manual alignment, this notebook is most useful for inverse and chain align with intermed. It is also most useful to visualize blocks with too many sections

In [ ]:
from histreg.manual_alignment import ManualBlockAligner, decompose_affine_matrix, inverse_affine_transform, make3x3
import pandas as pd

In [ ]:
cf = {                                                          
    "infra":{                                                           
        "annot_root": "/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/playgrounds/pixel_alignment/sections_annot2/",
        "svs_root": "/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/svs"
    },                                                                  
    "prev_exp": {                                                       
        "out_dir": "/nfs/turbo/umms-tocho-snr/exp/chengjia/block_align_manual" # block_align_crop_rigid_ransac"
    },                                                                  
    "manual": {                                                         
        "out_dir": "/nfs/turbo/umms-tocho-snr/exp/chengjia/block_align_manual"
    }                                                                   
}           

In [ ]:
aligner = ManualBlockAligner(cf)
aligner.init_one_block(block="SU-17-1909.D2")
aligner.commit_save_block(do_save=False)

In [ ]:
pd.set_option('display.float_format', lambda x: "%.1f" % x)
[pd.DataFrame([decompose_affine_matrix(j) for j in i]) for i in aligner.align_results["matrices"]]

### Chain align with intermed

In [ ]:
for hei in [1]:
    for ihci in range(2,18):
        aligner.set_align_pair(he_i=hei, ihc_i=ihci, do_viz=False)
        aligner.chain_align_with_intermed(intermed_i=0)
        aligner.commit_slide_pair(do_viz=True)

In [ ]:
aligner.commit_save_block(do_save=True)

### Visualize only

In [ ]:
for j in [0,1,2]:
    for i in range(3,52):
        aligner.set_align_pair(he_i=j, ihc_i=i, do_viz=True)